# Dominant class by boundary

Summarise the dominant categorical raster class within each boundary using cell counts.


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.features import rasterize

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
DATA = ROOT / 'data_raw'
OUTPUTS = ROOT / 'outputs'
TABLES = ROOT / 'tables'
for folder in (OUTPUTS, TABLES):
    folder.mkdir(parents=True, exist_ok=True)

BOUNDARY_FILE = DATA / 'boundary.gpkg'
RECLASSIFIED_RASTER = OUTPUTS / 'categorical_raster_reclassified.tif'
BOUNDARY_ID_FIELD = None
CLASS_LABELS = {}

if not BOUNDARY_FILE.exists():
    raise FileNotFoundError(f'Boundary file not found: {BOUNDARY_FILE}')
if not RECLASSIFIED_RASTER.exists():
    raise FileNotFoundError(f'Reclassified raster not found: {RECLASSIFIED_RASTER}')

boundaries = gpd.read_file(BOUNDARY_FILE)
with rasterio.open(RECLASSIFIED_RASTER) as src:
    if boundaries.crs != src.crs:
        boundaries = boundaries.to_crs(src.crs)
    class_arr = src.read(1, masked=False)
    nodata = src.nodata
    valid_codes = sorted(int(x) for x in pd.unique(class_arr.ravel()) if nodata is None or x != nodata)
    rows = []
    for i, geom in enumerate(boundaries.geometry):
        boundary_id = str(boundaries.loc[i, BOUNDARY_ID_FIELD]) if BOUNDARY_ID_FIELD and BOUNDARY_ID_FIELD in boundaries.columns else f'boundary_{i+1:03d}'
        mask = rasterize([(geom, 1)], out_shape=class_arr.shape, transform=src.transform, fill=0, all_touched=False, dtype='uint8').astype(bool)
        values = class_arr[mask]
        if nodata is not None:
            values = values[values != nodata]
        counts = pd.Series(values).value_counts()
        for code in valid_codes:
            rows.append({'boundary_id': boundary_id, 'class_code': int(code), 'class_label': CLASS_LABELS.get(int(code), 'unlabelled'), 'cell_count': int(counts.get(code, 0))})

result = pd.DataFrame(rows)
totals = result.groupby('boundary_id')['cell_count'].transform('sum')
result['cell_proportion'] = result['cell_count'] / totals.where(totals != 0, pd.NA)
dominant = result.sort_values(['boundary_id', 'cell_count'], ascending=[True, False]).drop_duplicates('boundary_id')
result.to_csv(TABLES / 'class_cell_counts_by_boundary.csv', index=False)
dominant.to_csv(TABLES / 'dominant_class_by_boundary.csv', index=False)
with open(OUTPUTS / 'dominant_class_summary.json', 'w', encoding='utf-8') as f:
    json.dump({'run_time_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'), 'n_boundaries': int(dominant['boundary_id'].nunique())}, f, indent=2)
dominant
